In [1]:
import os
import pandas

# Construction du csv emotion

In [2]:
FOLDER_EMOTION = "sentence_level_all_features"
FOLDER_EMOTION_UTF8 = "sentence_level_all_features_utf8"
NO_FEATURES_MESSAGE = "No features found."

In [3]:
# Conversion des fichiers MBCS -> UTF-8
os.makedirs(FOLDER_EMOTION_UTF8, exist_ok=True)

converted = 0
errors = []
for filename in os.listdir(FOLDER_EMOTION):
    if not filename.endswith(".txt"):
        continue
    src_path = os.path.join(FOLDER_EMOTION, filename)
    dst_path = os.path.join(FOLDER_EMOTION_UTF8, filename)
    try:
        with open(src_path, "r", encoding="mbcs") as f:
            content = f.read()
        with open(dst_path, "w", encoding="utf-8") as f:
            f.write(content)
        converted += 1
    except Exception as e:
        errors.append((filename, str(e)))

print(f"{converted} fichier(s) converti(s) en UTF-8.")
if errors:
    print(f"{len(errors)} erreur(s) :", errors)

2896 fichier(s) converti(s) en UTF-8.


In [4]:
def extract_text(lines):
    txt = ("\n").join(lines[4:-3])
    return txt.split("Sentence: ")[1].strip()

def extract_cats(lines):
    return lines[-1].split(": ")[1].replace("[", "").replace("]", "").replace(" ", "").split(",")

def extract_tokens(lines):
    return lines[-3].split(": ")[1].strip().replace("[", "").replace("]", "").replace(" ", "").split(",")

def extract_lemmas(lines):
    return lines[-2].split(": ")[1].strip().replace("[", "").replace("]", "").replace(" ", "").split(",")

In [5]:
def extract_from_file(lines,id):

    if lines[-1] == NO_FEATURES_MESSAGE:
        print("No features found for file id: ", id)
        return {"text": ("\n").join(lines[4:-1]).split("Sentence: ")[1].strip(),
                "Tokens": [],
                "Lemmas": [],
                "Cats": [],
                "file_id": id,
                "feature_found": False}
    
    else:
        return {"text": extract_text(lines),
                "Tokens": extract_tokens(lines),
                "Lemmas": extract_lemmas(lines),
                "Cats": extract_cats(lines),
                "file_id": id,
                "feature_found": True}
    

In [6]:
rows = []
for filename in os.listdir(FOLDER_EMOTION_UTF8):
    if filename.endswith(".txt"):
        path = os.path.join(FOLDER_EMOTION_UTF8, filename)
        with open(path, "r", encoding="utf-8") as txt:
            lines = txt.read().rstrip().split("\n")
        rows.append(extract_from_file(lines, int(filename.split(".")[0])))

print(len(rows), "rows extracted.")

No features found for file id:  100
No features found for file id:  1001
No features found for file id:  1003
No features found for file id:  1004
No features found for file id:  1005
No features found for file id:  1008
No features found for file id:  1009
No features found for file id:  101
No features found for file id:  1010
No features found for file id:  1011
No features found for file id:  1012
No features found for file id:  1013
No features found for file id:  1014
No features found for file id:  1016
No features found for file id:  1017
No features found for file id:  1019
No features found for file id:  102
No features found for file id:  1020
No features found for file id:  1021
No features found for file id:  1022
No features found for file id:  1023
No features found for file id:  1024
No features found for file id:  1025
No features found for file id:  1026
No features found for file id:  1027
No features found for file id:  1029
No features found for file id:  103
No fe

In [7]:
df = pandas.DataFrame(rows)

In [8]:
df.columns

Index(['text', 'Tokens', 'Lemmas', 'Cats', 'file_id', 'feature_found'], dtype='object')

In [9]:
df['feature_found'].value_counts()

feature_found
False    2404
True      492
Name: count, dtype: int64

In [10]:
df = df.set_index("file_id")
df = df.sort_index()
df.head()

,text,Tokens,Lemmas,Cats,feature_found
file_id,,,,,
1,"C'est dingue, après tout ce temps, plus de 15 ...",[rire],[rire],[joie],True
2,"Plutôt d'accord pour les SAV, c'est le but mêm...",[],[],[],False
3,Bordel.,[],[],[],False
4,J'avais tapé une longue réponse.,[],[],[],False
5,"Une fausse manip sur le téléphone, et hop, per...",[],[],[],False


In [11]:
df.to_csv("corpus_emotion_reddit_pc.csv", index=True)